# Módulo 5 — Queries Avançadas no Snowflake

Snowflake suporta SQL avançado com CTEs, window functions e funções de série temporal. Neste notebook exploramos padrões de query essenciais para análise de qualidade de dados.

## O que vamos ver

1. CTEs — organização de queries complexas
2. Window functions — ranking e análises comparativas
3. Queries de série temporal para tendências de consumo
4. Queries parametrizadas seguras

---

In [ ]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv("../../.env")

# Dados locais para simulação
df = pd.read_csv("../modulo2_dados_com_pandas/dados/consumidores_simulado.csv")
df["dat_ultima_leitura"] = pd.to_datetime(df["dat_ultima_leitura"], errors="coerce")
print(f"Dados locais: {len(df)} registros")

## 1. CTEs — Organização de Queries Complexas

CTEs (Common Table Expressions) tornam queries complexas legíveis ao nomear subconjuntos intermediários.

In [ ]:
# Query com múltiplas CTEs para relatório completo de qualidade
QUERY_QUALIDADE_CTE = """
-- Relatório de qualidade com CTEs encadeadas
WITH
-- CTE 1: Base de consumidores ativos
consumidores_ativos AS (
    SELECT *
    FROM CADASTRO.CONSUMIDORES_UC
    WHERE flg_ativo = TRUE
),

-- CTE 2: Calcular problemas por registro
problemas_por_uc AS (
    SELECT
        cod_consumidor,
        nom_consumidor,
        nom_municipio,
        cod_uf,
        cod_classe_consumo,

        -- Flags de problema (1 = tem problema)
        CASE WHEN cpf_cnpj IS NULL THEN 1 ELSE 0 END AS sem_cpf,
        CASE WHEN num_medidor IS NULL THEN 1 ELSE 0 END AS sem_medidor,
        CASE WHEN dat_ultima_leitura IS NULL THEN 1 ELSE 0 END AS sem_leitura,
        CASE WHEN num_cep IS NULL THEN 1 ELSE 0 END AS sem_cep,
        CASE WHEN num_latitude IS NULL THEN 1 ELSE 0 END AS sem_coordenada,
        CASE
            WHEN vlr_consumo_medio_kwh IS NULL OR vlr_consumo_medio_kwh < 0
                OR vlr_consumo_medio_kwh > 500000 THEN 1
            ELSE 0
        END AS consumo_invalido
    FROM consumidores_ativos
),

-- CTE 3: Score total por UC
score_por_uc AS (
    SELECT
        cod_consumidor,
        nom_consumidor,
        nom_municipio,
        cod_uf,
        cod_classe_consumo,
        sem_cpf + sem_medidor + sem_leitura + sem_cep + sem_coordenada + consumo_invalido
            AS qtd_problemas,
        CASE
            WHEN sem_cpf + sem_medidor + sem_leitura + sem_cep + sem_coordenada + consumo_invalido = 0
                THEN 'SEM_PROBLEMAS'
            WHEN sem_cpf + sem_medidor + sem_leitura + sem_cep + sem_coordenada + consumo_invalido <= 2
                THEN 'PROBLEMAS_MENORES'
            ELSE 'PROBLEMAS_CRITICOS'
        END AS categoria_qualidade
    FROM problemas_por_uc
),

-- CTE 4: Resumo por município
resumo_municipio AS (
    SELECT
        nom_municipio,
        cod_uf,
        COUNT(*) AS total_ucs,
        SUM(CASE WHEN categoria_qualidade = 'SEM_PROBLEMAS' THEN 1 ELSE 0 END) AS ucs_ok,
        SUM(CASE WHEN categoria_qualidade = 'PROBLEMAS_CRITICOS' THEN 1 ELSE 0 END) AS ucs_criticas,
        ROUND(SUM(CASE WHEN categoria_qualidade = 'SEM_PROBLEMAS' THEN 1 ELSE 0 END)
            / COUNT(*) * 100, 1) AS pct_ok
    FROM score_por_uc
    GROUP BY nom_municipio, cod_uf
)

-- Resultado final: municípios com mais UCs críticas
SELECT *
FROM resumo_municipio
WHERE ucs_criticas > 0
ORDER BY ucs_criticas DESC, pct_ok ASC
LIMIT 20;
"""

print("Query CTE preparada (para execução no Snowflake).")
print(f"Comprimento: {len(QUERY_QUALIDADE_CTE)} caracteres")

# Simular resultado equivalente em pandas
print("\n--- Equivalente em pandas ---")

df["qtd_problemas"] = (
    df["cpf_cnpj"].isnull().astype(int) +
    df["num_medidor"].isnull().astype(int) +
    df["dat_ultima_leitura"].isnull().astype(int) +
    df["num_cep"].isnull().astype(int) +
    (df["vlr_consumo_medio_kwh"] > 500000).astype(int)
)

resumo_municipio = (
    df.groupby("nom_municipio")
    .agg(
        total_ucs=("cod_consumidor", "count"),
        ucs_sem_problemas=("qtd_problemas", lambda x: (x == 0).sum()),
        ucs_criticas=("qtd_problemas", lambda x: (x > 2).sum()),
    )
    .assign(pct_ok=lambda x: (x["ucs_sem_problemas"] / x["total_ucs"] * 100).round(1))
    .sort_values("ucs_criticas", ascending=False)
    .reset_index()
)

print(resumo_municipio[resumo_municipio["ucs_criticas"] > 0].to_string(index=False))

## 2. Window Functions — Ranking de UCs por Problemas

In [ ]:
# Window function: ranking de UCs por % de nulos dentro de cada município
QUERY_WINDOW = """
-- Ranking de municípios por % de UCs sem CPF/CNPJ
-- Usando window functions para calcular percentual e rank
WITH base AS (
    SELECT
        nom_municipio,
        cod_uf,
        COUNT(*) AS total_ucs,
        SUM(CASE WHEN cpf_cnpj IS NULL THEN 1 ELSE 0 END) AS sem_cpf
    FROM CADASTRO.CONSUMIDORES_UC
    WHERE flg_ativo = TRUE
    GROUP BY nom_municipio, cod_uf
    HAVING COUNT(*) >= 5  -- mínimo de 5 UCs para ser relevante
)
SELECT
    nom_municipio,
    cod_uf,
    total_ucs,
    sem_cpf,
    ROUND(sem_cpf / total_ucs * 100, 1) AS pct_sem_cpf,

    -- Ranking global
    RANK() OVER (
        ORDER BY ROUND(sem_cpf / total_ucs * 100, 1) DESC
    ) AS rank_global,

    -- Ranking dentro do estado (UF)
    RANK() OVER (
        PARTITION BY cod_uf
        ORDER BY ROUND(sem_cpf / total_ucs * 100, 1) DESC
    ) AS rank_na_uf,

    -- Percentual acumulado de UCs com problema (running total)
    SUM(sem_cpf) OVER (
        ORDER BY ROUND(sem_cpf / total_ucs * 100, 1) DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS acumulado_sem_cpf

FROM base
WHERE sem_cpf > 0
ORDER BY pct_sem_cpf DESC
LIMIT 20;
"""

print("Query com window functions preparada.")

# Equivalente em pandas
print("\n--- Equivalente em pandas ---")

wf_data = (
    df.groupby(["nom_municipio", "cod_uf"])
    .agg(
        total_ucs=("cod_consumidor", "count"),
        sem_cpf=("cpf_cnpj", lambda x: x.isnull().sum())
    )
    .reset_index()
    .assign(pct_sem_cpf=lambda x: (x["sem_cpf"] / x["total_ucs"] * 100).round(1))
    .sort_values("pct_sem_cpf", ascending=False)
)

# Rank global
wf_data["rank_global"] = wf_data["pct_sem_cpf"].rank(method="min", ascending=False).astype(int)

# Rank por UF
wf_data["rank_na_uf"] = (
    wf_data.groupby("cod_uf")["pct_sem_cpf"]
    .rank(method="min", ascending=False)
    .astype(int)
)

print(wf_data[wf_data["sem_cpf"] > 0][["nom_municipio", "cod_uf", "total_ucs", "sem_cpf", "pct_sem_cpf", "rank_global"]].to_string(index=False))

## 3. Série Temporal — Tendências de Consumo

In [ ]:
# Query de série temporal (assumindo tabela de histórico de leituras)
QUERY_SERIE_TEMPORAL = """
-- Tendência mensal de consumo médio por classe
-- Tabela: CADASTRO.HISTORICO_LEITURAS
-- Colunas: cod_consumidor, dat_leitura, vlr_leitura_kwh, cod_classe_consumo

WITH leituras_mensais AS (
    SELECT
        DATE_TRUNC('MONTH', dat_leitura) AS mes,
        cod_classe_consumo,
        COUNT(*) AS qtd_leituras,
        AVG(vlr_leitura_kwh) AS consumo_medio,
        SUM(vlr_leitura_kwh) AS consumo_total
    FROM CADASTRO.HISTORICO_LEITURAS
    WHERE dat_leitura >= DATEADD('MONTH', -12, CURRENT_DATE())
    GROUP BY DATE_TRUNC('MONTH', dat_leitura), cod_classe_consumo
),

-- Variação mês a mês usando LAG (window function)
com_variacao AS (
    SELECT
        mes,
        cod_classe_consumo,
        qtd_leituras,
        ROUND(consumo_medio, 1) AS consumo_medio,
        ROUND(consumo_total, 0) AS consumo_total,

        -- Consumo do mês anterior (LAG)
        LAG(consumo_medio) OVER (
            PARTITION BY cod_classe_consumo
            ORDER BY mes
        ) AS consumo_medio_mes_anterior,

        -- Variação percentual
        ROUND(
            (consumo_medio - LAG(consumo_medio) OVER (
                PARTITION BY cod_classe_consumo ORDER BY mes
            ))
            / NULLIF(LAG(consumo_medio) OVER (
                PARTITION BY cod_classe_consumo ORDER BY mes
            ), 0) * 100,
        1) AS variacao_pct
    FROM leituras_mensais
)

SELECT *
FROM com_variacao
WHERE mes >= DATEADD('MONTH', -6, CURRENT_DATE())
ORDER BY cod_classe_consumo, mes;
"""

print("Query de série temporal com window functions preparada.")
print("\nElementos da query:")
print("  - DATE_TRUNC: agrupa por mês")
print("  - DATEADD: filtra últimos 12 meses")
print("  - LAG() OVER (PARTITION BY ... ORDER BY ...): acessa valor do mês anterior")
print("  - NULLIF: evita divisão por zero na variação percentual")

# Simular série temporal com dados gerados
print("\n--- Série temporal simulada (equivalente pandas) ---")

np.random.seed(42)
meses = pd.date_range("2024-08", "2025-02", freq="MS")
classes = ["RESIDENCIAL", "COMERCIAL", "INDUSTRIAL"]
base_consumo = {"RESIDENCIAL": 280, "COMERCIAL": 4500, "INDUSTRIAL": 80000}

serie_simulada = []
for classe in classes:
    consumo_anterior = None
    for mes in meses:
        consumo = base_consumo[classe] * (1 + np.random.normal(0, 0.08))
        variacao = None if consumo_anterior is None else round(
            (consumo - consumo_anterior) / consumo_anterior * 100, 1
        )
        serie_simulada.append({
            "mes": mes.strftime("%Y-%m"),
            "classe": classe,
            "consumo_medio": round(consumo, 1),
            "variacao_pct": variacao
        })
        consumo_anterior = consumo

df_serie = pd.DataFrame(serie_simulada)
print(df_serie[df_serie["classe"] == "RESIDENCIAL"].to_string(index=False))

## 4. Queries Parametrizadas — Segurança contra SQL Injection

In [ ]:
# Demonstração: por que usar queries parametrizadas

# INSEGURO — nunca faça isso com inputs do usuário
def query_insegura(uf: str, limite: int) -> str:
    """EXEMPLO DO QUE NÃO FAZER — vulnerável a SQL injection."""
    return f"SELECT * FROM CONSUMIDORES_UC WHERE cod_uf = '{uf}' LIMIT {limite}"

# Problema: se uf receber "SP' OR '1'='1" — retorna TUDO
uf_maliciosa = "SP' OR '1'='1"
query_gerada = query_insegura(uf_maliciosa, 10)
print("Query INSEGURA gerada:")
print(query_gerada)
print("\nProblema: a condição '1'='1' sempre é verdadeira — retorna todos os dados!")

print()

# SEGURO — use parâmetros com %s
def query_segura_parametros() -> tuple:
    """Retorna query com placeholders e parâmetros separados."""
    query = """
        SELECT cod_consumidor, nom_consumidor, vlr_consumo_medio_kwh
        FROM CONSUMIDORES_UC
        WHERE cod_uf = %s
          AND cod_classe_consumo = %s
          AND flg_ativo = TRUE
        ORDER BY vlr_consumo_medio_kwh DESC
        LIMIT %s
    """
    # Parâmetros separados — o driver trata o escaping automaticamente
    params = ("SP", "RESIDENCIAL", 100)
    return query, params

query, params = query_segura_parametros()
print("Query SEGURA com parâmetros:")
print(query.strip())
print(f"\nParâmetros: {params}")

print()
print("Uso com snowflake-connector-python:")
print("""
with SnowflakeConector() as sf:
    # O conector faz o escape dos parâmetros automaticamente
    df = sf.query_para_dataframe(query, params=params)
""")

In [ ]:
# Resumo das técnicas avançadas de SQL
tecnicas = [
    {
        "tecnica": "CTE (WITH)",
        "uso": "Dividir queries complexas em partes nomeadas e legíveis",
        "quando_usar": "Query com 3+ passos lógicos ou subqueries repetidas",
        "exemplo": "WITH base AS (...), resumo AS (...) SELECT * FROM resumo"
    },
    {
        "tecnica": "RANK() / ROW_NUMBER()",
        "uso": "Criar ranking de registros por algum critério",
        "quando_usar": "Top-N por grupo, pior/melhor por município",
        "exemplo": "RANK() OVER (PARTITION BY cod_uf ORDER BY pct_nulos DESC)"
    },
    {
        "tecnica": "LAG() / LEAD()",
        "uso": "Acessar valores de linhas anteriores/seguintes",
        "quando_usar": "Calcular variação mês a mês, detectar mudanças de valor",
        "exemplo": "LAG(consumo) OVER (PARTITION BY classe ORDER BY mes)"
    },
    {
        "tecnica": "DATE_TRUNC / DATEADD",
        "uso": "Manipulação de datas em análises temporais",
        "quando_usar": "Agrupamento por mês/ano, filtros por período",
        "exemplo": "DATE_TRUNC('MONTH', dat_leitura)"
    },
    {
        "tecnica": "Queries parametrizadas",
        "uso": "Passar valores de forma segura sem SQL injection",
        "quando_usar": "Sempre que valores vierem de variáveis Python",
        "exemplo": "cursor.execute('SELECT ... WHERE uf = %s', ('SP',))"
    },
]

print("Resumo das técnicas avançadas de SQL para análise de dados:")
print()
for t in tecnicas:
    print(f"[{t['tecnica']}]")
    print(f"  Uso: {t['uso']}")
    print(f"  Quando usar: {t['quando_usar']}")
    print(f"  Exemplo: {t['exemplo']}")
    print()